# Same pipeline, no PyTorch

In [1]:
%load_ext autoreload
%autoreload 2

First, comparing queries against a document:

In [2]:
from src.embed.embedder import Embedder

embed = Embedder("models/Xenova/bge-base-en-v1.5")

q1 = 'Can I still join the course after the start date?'
q2 = 'How to install Docker on Windows?'
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [3]:
v1.dot(dv), v2.dot(dv)

(np.float64(0.6628812385282712), np.float64(0.4017860018225564))

Load the documents:

In [4]:
from src.ingest import FaqHttpLoader

loader = FaqHttpLoader()

documents = loader.load()

In [5]:
texts = [doc['question'] + ' ' + doc['answer'] for doc in documents]

Embed in batches:

In [6]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/25 [00:00<?, ?it/s]

search:

In [7]:
query = 'Can I still join the course after the start date?'
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

print(f"Best match score: {scores[idx]:.1%}")
documents[idx]

Best match score: 82.9%


{'question': 'Course - Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.",
 'section': 'General Course-Related Questions',
 'course': 'mlops-zoomcamp'}